In [4]:
# ------------------------------------------------------------------------------
# 1. Import Required Packages
# ------------------------------------------------------------------------------
import dask
import dask.dataframe as dd
from dask.distributed import Client
import hvplot.dask
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
import pandas as pd
import pprint
import requests

import datetime
from io import StringIO

In [5]:
selected_12 = pd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12.parquet')

# Read the file for urban factors, SWORD reaches, usgs gage heights
sword_urban = gpd.read_parquet(
    "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/usgs_sword_matchups.parquet"
)

sword_urban = sword_urban.dropna(axis=1, how='all')

In [8]:
selected_12

,reach_id,reach_len,wse,wse_var,width,width_var,facc,n_chan_max,n_chan_mod,obstr_type,...,nlcd_21_mean,nlcd_22_mean,nlcd_23_mean,nlcd_24_mean,fctimp_count,fctimp_mean,site_no,percentile,site_number,target_value
55364,73270200201,18868.560346,24.200001,1.143635,67.0,924.576612,3.736451e+03,1,1,0,...,0.022249,0.006893,0.001804,0.000043,13018.079345,0.005496,02361500,1.0,1,0.005481
137641,74282700071,18861.835615,148.100006,0.746737,204.0,1112.908008,1.742668e+04,2,1,0,...,0.021400,0.051541,0.036223,0.013228,42497.589118,0.053897,05542500,10.0,2,0.054807
238014,78261200051,17463.498956,146.900009,34.055258,63.0,522.785243,1.501463e+04,2,1,0,...,0.045410,0.205395,0.052549,0.000165,12233.567468,0.104614,12510500,19.0,3,0.104134
243162,78264000141,19445.059172,510.300018,119.197919,48.0,384.908050,1.218250e+04,1,1,0,...,0.136858,0.222216,0.087937,0.005034,10352.709821,0.148739,12422500,28.0,4,0.153461
133613,74282600011,8782.559445,146.000000,3.947155,108.0,2372.359149,5.578036e+03,2,1,0,...,0.103785,0.173330,0.139307,0.040523,10590.777721,0.193211,05552500,37.0,5,0.202787
224666,75140800061,1918.010718,126.599998,0.000000,48.0,3193.157216,6.577440e+03,1,1,0,...,0.073499,0.287961,0.200466,0.023184,1049.335909,0.250249,08055560,46.0,6,0.252114
140706,74282700211,10961.543680,170.600006,12.587271,288.0,27281.576464,4.254222e+02,7,2,0,...,0.055894,0.147256,0.200123,0.135900,35269.033734,0.304098,05533600,55.0,7,0.301441
225701,75140900171,8613.945067,160.400009,0.842093,48.0,2104.110433,6.530376e+03,4,1,0,...,0.255677,0.361852,0.141233,0.133527,4602.381284,0.356397,08048000,64.0,8,0.350767
139708,74282700201,11106.375378,156.600006,7.454049,195.0,13531.591541,5.166817e+02,7,2,0,...,0.043134,0.178600,0.317140,0.115752,24232.932720,0.373023,05537980,73.0,9,0.400094
76973,73120000121,18201.098440,11.400001,17.519935,283.0,3330.788878,2.158350e+04,2,1,0,...,0.171900,0.224902,0.270938,0.167597,58913.370424,0.412258,01184000,82.0,10,0.449421


In [9]:
# ------------------------------------------------------------------------------
# 2. Set Up Dask Client
# ------------------------------------------------------------------------------
client = Client(n_workers=48)  # Create local Dask client
client             # Display client information (includes dashboard link)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 48
Total threads: 48,Total memory: 256.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33075,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:35261,Total threads: 1
Dashboard: http://127.0.0.1:36153/status,Memory: 5.33 GiB
Nanny: tcp://127.0.0.1:37409,


In [12]:
# Assign URLs to Variables for the APIs we use, FTS and Hydrocron
FTS_URL = "https://fts.podaac.earthdata.nasa.gov/v1"  
HYDROCRON_URL = "https://soto.podaac.earthdatacloud.nasa.gov/hydrocron/v1/timeseries"

# BASIN or RIVER to query FTS for
BASIN_IDENTIFIER = "732520" # to search via basin ID, find within SWORD database
RIVER_NAME = "Rhine" # to search via river name

In [13]:
def query_fts(query_url, params):
    """Query Feature Translation Service (FTS) for reach identifers using the query_url parameter.

    Parameters
    ----------
    query_url: str - URL to use to query FTS
    params: dict - Dictionary of parameters to pass to query

    Returns
    -------
    dict of results: hits, page_size, page_number, reach_ids
    """

    reaches = requests.get(query_url, params=params)
    reaches_json = reaches.json()

    hits = reaches_json['hits']
    if 'search on' in reaches_json.keys():
        page_size = reaches_json['search on']['page_size']
        page_number = reaches_json['search on']['page_number']
    else:
        page_size = 0
        page_number = 0

    return {
        "hits": hits,
        "page_size": page_size,
        "page_number": page_number,
        "reach_ids": [ item['reach_id'] for item in reaches_json['results'] ]
    }

In [14]:
# Search by basin code
query_url = f"{FTS_URL}/rivers/reach/{BASIN_IDENTIFIER}"
print(f"Searching by basin ...{query_url}")

# Search by river name
# query_url = f"{FTS_URL}/rivers/{RIVER_NAME}" #if searching via river name instead
# print(f"Searching by river name ...{query_url}")

page_size = 100    # Set FTS to retrieve 100 results at a time
page_number = 1    # Set FTS to retrieve the first page of results
hits = 1           # Set hits to intial value to start while loop
reach_ids = []
while (page_size * page_number) != 0 and len(reach_ids) < hits:
    params = { "page_size": page_size, "page_number": page_number }
    results = query_fts(query_url, params)
    
    hits = results['hits']
    page_size = results['page_size']
    page_number = results['page_number'] + 1
    reach_ids.extend(results['reach_ids'])

    print("page_size: ", page_size, ", page_number: ", page_number - 1, ", hits: ", hits, ", # reach_ids: ", len(reach_ids))
    
print("Total number of reaches: ", len(reach_ids))
reach_ids = list(set(reach_ids))    # Remove duplicates
print("Total number of non-duplicate reaches: ", len(reach_ids))

Searching by basin ...https://fts.podaac.earthdata.nasa.gov/v1/rivers/reach/732520
page_size:  100 , page_number:  1 , hits:  190 , # reach_ids:  100
page_size:  100 , page_number:  2 , hits:  190 , # reach_ids:  190
Total number of reaches:  190
Total number of non-duplicate reaches:  190


In [15]:
reach_ids

['73252001271',
 '73252001776',
 '73252001033',
 '73252000763',
 '73252000843',
 '73252000101',
 '73252001726',
 '73252000511',
 '73252001736',
 '73252000613',
 '73252001916',
 '73252000903',
 '73252000403',
 '73252000953',
 '73252001123',
 '73252000973',
 '73252001796',
 '73252001786',
 '73252000171',
 '73252001363',
 '73252001866',
 '73252001573',
 '73252000604',
 '73252001625',
 '73252000863',
 '73252001483',
 '73252001203',
 '73252000161',
 '73252000873',
 '73252001131',
 '73252001263',
 '73252000391',
 '73252000643',
 '73252001151',
 '73252000833',
 '73252000853',
 '73252001605',
 '73252001665',
 '73252001341',
 '73252000464',
 '73252001896',
 '73252000963',
 '73252001404',
 '73252001716',
 '73252000201',
 '73252001421',
 '73252001504',
 '73252001633',
 '73252001331',
 '73252000943',
 '73252000273',
 '73252000263',
 '73252000933',
 '73252000141',
 '73252000111',
 '73252001171',
 '73252001241',
 '73252001001',
 '73252001193',
 '73252000353',
 '73252000633',
 '73252000051',
 '732520

In [8]:
reach_ids = reach_ids[:10]
len(reach_ids)

10

In [9]:
@dask.delayed
def query_hydrocron(query_url, reach_id, start_time, end_time, fields, empty_df):
    """Query Hydrocron for reach-level time series data.

    Parameters
    ----------
    query_url: str - URL to use to query FTS
    reach_id: str - String SWORD reach identifier
    start_time: str - String time to start query
    end_time: str - String time to end query
    fields: list - List of fields to return in query response
    empty_df: pandas.DataFrame that contains empty query results

    Returns
    -------
    pandas.DataFrame that contains query results
    """

    params = {
        "feature": "Reach",
        "feature_id": reach_id,
        "output": "csv",
        "start_time": start_time,
        "end_time": end_time,
        "fields": fields
    }
    results = requests.get(query_url, params=params)
    if "results" in results.json().keys():
        results_csv = results.json()["results"]["csv"]
        df = pd.read_csv(StringIO(results_csv))
    else:
        df = empty_df

    return df

In [10]:
%%time
# Create queries that return Pandas.DataFrame objects
start_time = "2023-07-28T00:00:00Z"
end_time = "2024-04-16T00:00:00Z"
fields = "reach_id,time_str,wse"
results = []
for reach in reach_ids:
    # Create an empty dataframe for cases where no data is returned for a reach identifier
    empty_df = pd.DataFrame({
        "reach_id": np.int64(reach),
        "time_str": datetime.datetime(1900, 1, 1).strftime("%Y-%m-%dT%H:%M:%S"),
        "wse": -999999999999.0,
        "wse_units": "m"
    }, index=[0])
    results.append(query_hydrocron(HYDROCRON_URL, reach, start_time, end_time, fields, empty_df))

# Load DataFrame results into dask.dataframe
ddf = dd.from_delayed(results)
ddf.head(n=20, npartitions=len(reach_ids))

CPU times: user 561 ms, sys: 359 ms, total: 920 ms
Wall time: 6.36 s


,reach_id,time_str,wse,wse_units
0,73252000833,2023-08-05T20:31:32Z,226.1414,m
1,73252000833,2023-08-26T17:16:37Z,226.1816,m
2,73252000833,2023-09-06T04:50:52Z,226.2001,m
3,73252000833,2023-09-16T14:01:43Z,226.2785,m
4,73252000833,2023-10-07T10:46:44Z,226.2107,m
5,73252000833,2023-10-17T22:20:59Z,226.2678,m
6,73252000833,2023-10-28T07:31:50Z,226.2127,m
7,73252000833,2023-11-07T19:06:04Z,226.1365,m
8,73252000833,2023-11-18T04:16:55Z,228.6804,m
9,73252000833,2023-11-28T15:51:10Z,226.1444,m


In [12]:
# Remove fill values for missing observations
ddf = ddf.loc[(ddf["wse"] != -999999999999.0)]

# Convert time_str to datetime format
ddf.time_str = dd.to_datetime(ddf.time_str)

ddf.head(n=20, npartitions=len(reach_ids))

,reach_id,time_str,wse,wse_units
0,73252000833,2023-08-05 20:31:32+00:00,226.1414,m
1,73252000833,2023-08-26 17:16:37+00:00,226.1816,m
2,73252000833,2023-09-06 04:50:52+00:00,226.2001,m
3,73252000833,2023-09-16 14:01:43+00:00,226.2785,m
4,73252000833,2023-10-07 10:46:44+00:00,226.2107,m
5,73252000833,2023-10-17 22:20:59+00:00,226.2678,m
6,73252000833,2023-10-28 07:31:50+00:00,226.2127,m
7,73252000833,2023-11-07 19:06:04+00:00,226.1365,m
8,73252000833,2023-11-18 04:16:55+00:00,228.6804,m
9,73252000833,2023-11-28 15:51:10+00:00,226.1444,m


In [13]:
# Plot results
line_plot = ddf.hvplot(x="time_str", y="wse", by="reach_id", kind="line", persist=True)
line_plot.opts(xrotation=90)

scatter_plot = ddf.hvplot(x="time_str", y="wse", by="reach_id", kind="scatter", persist=True)
line_plot * scatter_plot

:Overlay
   .NdOverlay.I  :NdOverlay   [reach_id]
      :Curve   [time_str]   (wse)
   .NdOverlay.II :NdOverlay   [reach_id]
      :Scatter   [time_str]   (wse)

In [14]:
client.close()